# Random Forest (Text): Predicting `made_it_contract` from Scout Reports

We combine the **overview**, **strengths**, and **weaknesses** columns from `draft_enriched_with_contracts.csv`, pre-process the text, and train a TF-IDF + Random Forest pipeline — following the same approach as the course NLP application.

## 1. Imports

In [ ]:
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
from time import time

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold,
    cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

RANDOM_STATE = 42

: 

## Controls

In [ ]:
# ── Controls ──────────────────────────────────────────────────────────────────
# Draft year range (inclusive)
YEAR_MIN = 2014
YEAR_MAX = 2021

# Grade range (inclusive). Set to None to disable the bound.
GRADE_MIN = None  # e.g. 7.0 to restrict to highly-graded prospects
GRADE_MAX = 6.4

# N-gram range for CountVectorizer.
# (1,1) = unigrams only, (2,2) = bigrams only, (1,2) = uni+bi, (1,3) = uni+bi+tri
NGRAM_RANGE = (2, 3)

# Position filter — choose the aggregation level and optionally restrict to
# specific values at that level (set POS_FILTER = None to include all).
#   "position"  → raw position codes   e.g. ["QB"], ["WR", "TE"], ["CB", "S"]
#   "Pos_Group" → position groups      e.g. ["QB"], ["OL"], ["DB"], ["EDGE"]
#   "Group"     → broad groups         e.g. ["QB"], ["SKILL O"], ["BIG D"]
POS_LEVEL  = "Pos_Group"    # one of: "position", "Pos_Group", "Group"
POS_FILTER = None        # e.g. ["WR", "TE"] or None for all

# Feature type — controls which words are kept before TF-IDF.
# "all"        → all words, regex pre-processing only (original behaviour)
# "adjectives" → spaCy ADJ lemmas only  (trait descriptors, no leakage)
# "adj_adv"    → spaCy ADJ + ADV lemmas (adds intensity/frequency words)
FEATURE_TYPE = "all"

# Outcome-leaking PHRASES stripped from raw text before any processing.
# Longest phrases first so sub-phrases don't leave orphan tokens.
PHRASE_BLOCKLIST = {
    "undrafted free agent", "practice squad", "free agent", "early starter"
    "pro bowl", "late round", "undrafted free", "make roster", "rostered"
}

# Outcome-leaking single ADJECTIVES excluded after spaCy tagging.
# (Only applies when FEATURE_TYPE is "adjectives" or "adj_adv".)
ADJ_BLOCKLIST = {"undrafted", "rostered", "roster", "starter"}
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import subprocess, sys
import spacy

if FEATURE_TYPE != "all":
    _disable = ["parser", "ner"]
    try:
        nlp = spacy.load("en_core_web_sm", disable=_disable)
    except OSError:
        print("Downloading en_core_web_sm...")
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
        nlp = spacy.load("en_core_web_sm", disable=_disable)
    nlp.max_length = 2_000_000
    print(f"spaCy loaded (FEATURE_TYPE='{FEATURE_TYPE}')")
else:
    nlp = None
    print(f"spaCy not needed (FEATURE_TYPE='{FEATURE_TYPE}')")

## 2. Load & Combine Text

In [ ]:
df = pd.read_csv("../data/processed/draft_enriched_with_contracts.csv")

TARGET = "made_it_contract"

# ── Apply controls ─────────────────────────────────────────────────────────────
df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)].copy()
if GRADE_MIN is not None:
    df = df[df["grade"] >= GRADE_MIN]
if GRADE_MAX is not None:
    df = df[df["grade"] <= GRADE_MAX]
if POS_FILTER is not None:
    df = df[df[POS_LEVEL].isin(POS_FILTER)]
df = df.copy()

pos_desc = f"{POS_LEVEL}={POS_FILTER}" if POS_FILTER else "all positions"
print(f"Year: {YEAR_MIN}–{YEAR_MAX}  |  Grade: {GRADE_MIN or '—'}–{GRADE_MAX or '—'}  |  {pos_desc}")
print(f"Rows after filters: {len(df)}")

# Drop rows where target is missing
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

# Combine the three text columns; fill NaN with empty string before joining
df['combined_text'] = (
    df['overview'].fillna('') + ' ' +
    df['strengths'].fillna('') + ' ' +
    df['weaknesses'].fillna('')
).str.strip()

# Drop rows where all three columns were empty (no text at all)
df = df[df['combined_text'] != ''].copy()

print(f"Rows with text + target: {len(df)}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts())
print(f"\nPositive rate: {df[TARGET].mean():.1%}")

## 3. Text Pre-processing

Following the course approach: remove punctuation & digits, lowercase, remove stop-words, lemmatize.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Sort phrase blocklist longest-first so "undrafted free agent" is removed
# before "undrafted free" or "free agent" can leave orphan tokens.
_sorted_phrases = sorted(PHRASE_BLOCKLIST, key=len, reverse=True)

def remove_phrases(text: str) -> str:
    text = text.lower()
    for phrase in _sorted_phrases:
        text = text.replace(phrase, " ")
    return text

def preprocess_all(text: str) -> str:
    """Original path: all words, regex + NLTK lemmatize."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = remove_phrases(text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 1
    ]
    return ' '.join(tokens)

def preprocess_spacy_batch(texts: list[str], pos_tags: set[str]) -> list[str]:
    """spaCy path: keep only tokens whose POS is in pos_tags."""
    cleaned = [remove_phrases(t) if isinstance(t, str) else '' for t in texts]
    results = []
    for doc in nlp.pipe(cleaned, batch_size=64):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if token.pos_ in pos_tags
            and token.lemma_.lower() not in ADJ_BLOCKLIST
            and not token.is_stop
            and token.is_alpha
            and len(token.lemma_) > 1
        ]
        results.append(' '.join(tokens))
    return results

# Apply preprocessing
if FEATURE_TYPE == "all":
    df['text_preproc'] = df['combined_text'].apply(preprocess_all)
    print(f"Mode: all words (regex + NLTK)")
else:
    pos_tags = {"ADJ"} if FEATURE_TYPE == "adjectives" else {"ADJ", "ADV"}
    df['text_preproc'] = preprocess_spacy_batch(df['combined_text'].tolist(), pos_tags)
    print(f"Mode: spaCy {FEATURE_TYPE} (POS tags: {pos_tags})")

df = df[df['text_preproc'] != ''].copy().reset_index(drop=True)

print(f"Docs after preprocessing: {len(df)}")
print("\nSample pre-processed text:")
print(df['text_preproc'].iloc[0])

## 4. Vocabulary Exploration

Check how vocabulary size changes with different `min_df` / `max_df` cutoffs — same exercise as in the course.

In [ ]:
Texts = df['text_preproc'].tolist()
Labels = df[TARGET].tolist()

print(f"Total documents: {len(Texts)}")
print()

# No cutoff
cv0 = CountVectorizer(ngram_range=(1, 2))
cv0.fit(Texts)
print(f"No cutoff — vocab size: {len(cv0.vocabulary_):,}")

# Various cutoffs
for min_df, max_df in [(0.001, 1.0), (0.005, 0.9), (0.01, 0.8), (0.02, 0.7)]:
    cv_tmp = CountVectorizer(ngram_range=(1, 2), min_df=min_df, max_df=max_df)
    cv_tmp.fit(Texts)
    print(f"min_df={min_df}, max_df={max_df} — vocab size: {len(cv_tmp.vocabulary_):,}")

## 5. Pipeline: CountVectorizer → TF-IDF → Random Forest

Grid search over vectorizer and classifier hyperparameters, scored by **Average Precision (PR-AUC)** with 3-fold CV.

In [ ]:
pipeline = Pipeline(
    [
        ("vect", CountVectorizer()),
        ("tfidf", TfidfTransformer()),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
    ]
)

parameters = {
    "vect__max_df":          [0.5, 0.7],
    "vect__min_df":          [0.005],
    "vect__ngram_range":     [NGRAM_RANGE],
    "tfidf__use_idf":        [True],
    "tfidf__norm":           ['l2'],
    "clf__max_depth":        [10, 20],
    "clf__n_estimators":     [300],
    "clf__min_samples_leaf": [5, 10],
    "clf__class_weight":     ["balanced"],
}

grid_search = GridSearchCV(
    pipeline, parameters,
    n_jobs=-1, verbose=2,
    scoring='average_precision', cv=3
)

print("Performing grid search...")
pprint(parameters)
t0 = time()
grid_search.fit(Texts, Labels)
print(f"\nDone in {time() - t0:.1f}s")
print(f"\nBest PR-AUC (Average Precision): {grid_search.best_score_:.4f}")
print("Best parameters:")
best_params = grid_search.best_estimator_.get_params()
for p in sorted(parameters.keys()):
    print(f"  {p}: {best_params[p]}")

## 6. Evaluation

PR-AUC (Average Precision) is the right metric when we only care about identifying players who made it. A random classifier scores ≈ positive rate.

In [ ]:
best_clf = grid_search.best_estimator_

# Out-of-sample predicted probabilities via cross_val_predict
y_pred_proba = cross_val_predict(
    best_clf, Texts, Labels,
    cv=3, method='predict_proba', n_jobs=-1, verbose=2
)[:, 1]

y_pred = (y_pred_proba >= 0.5).astype(int)

baseline = np.mean(Labels)
pr_auc = average_precision_score(Labels, y_pred_proba)

print(classification_report(Labels, y_pred, target_names=['did not make it', 'made it']))
print(f"PR-AUC (Average Precision): {pr_auc:.4f}  |  Random baseline: {baseline:.4f}")

## 7. 5-Fold Cross-Validated PR-AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_clf, Texts, Labels, cv=cv, scoring='average_precision')
print(f"5-fold CV PR-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Fold scores: {cv_scores.round(4)}")

## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    Labels, y_pred,
    display_labels=['Did Not Make It', 'Made It'],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title('Confusion Matrix')

# Precision-Recall curve
PrecisionRecallDisplay.from_predictions(
    Labels, y_pred_proba,
    ax=axes[1],
    name='Random Forest (Text)'
)
axes[1].axhline(
    y=baseline, color='k', linestyle='--',
    label=f'Random baseline ({baseline:.2f})'
)
axes[1].set_title('Precision-Recall Curve (Made It)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Top Feature Importances

Refit the best model on the full dataset to extract feature importances.

In [ ]:
best_clf.fit(Texts, Labels)

feature_names = np.array(best_clf.named_steps['vect'].get_feature_names_out())
importances = best_clf.named_steps['clf'].feature_importances_

top_n = 30
indices = np.argsort(importances)[::-1][:top_n]

imp_series = pd.Series(importances[indices], index=feature_names[indices]).sort_values()

fig, ax = plt.subplots(figsize=(8, 9))
imp_series.plot.barh(ax=ax)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title(f'Top {top_n} Feature Importances (Text RF)')
plt.tight_layout()
plt.show()